# Chương 1 — Python nền tảng cho Trí tuệ kinh doanh

Notebook làm quen công cụ, chuẩn bị cho:

- **Chương 2:** làm sạch, tổng hợp, OLAP và dashboard mô tả;
- **Chương 3:** tạo biến, chia dữ liệu và đánh giá dự báo;
- **Chương 4:** tính giá trị kỳ vọng, ràng buộc và chọn phương án.

Đây không phải bài thực hành riêng về lý thuyết BI–BA–DSS. Tình huống ngắn chỉ
được dùng để luyện Python và tư duy **dữ liệu → phân tích → quyết định**.

> Dữ liệu mô phỏng phục vụ học tập. Trên Colab chọn **Runtime → Run all**.

## Chuẩn đầu ra

Sau notebook, sinh viên có thể:

1. Dùng biến, kiểu dữ liệu, điều kiện, vòng lặp và hàm trong Python.
2. Làm việc với `Series`/`DataFrame`, lọc và tổng hợp bằng Pandas.
3. Vẽ biểu đồ cơ bản và phân biệt mô tả với dự báo.
4. Viết phép tính giá trị kỳ vọng để chuẩn bị cho bài toán đề xuất.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_SEED = 42

## 1. Python cơ bản — biến, kiểu dữ liệu và biểu thức

Một KPI cần tên, giá trị, đơn vị và ngưỡng. Hãy chạy rồi thay đổi dữ liệu để
quan sát kết quả.

In [ ]:
revenue = 125_000_000       # int
profit = 21_500_000         # int
target_margin = 0.20        # float
period = "2026-Q2"          # str

profit_margin = profit / revenue
status = "ĐẠT" if profit_margin >= target_margin else "CẢNH BÁO"

print(f"Kỳ: {period}")
print(f"Doanh thu: {revenue:,.0f} đ")
print(f"Biên lợi nhuận: {profit_margin:.1%} — {status}")

### Bài tập 1

Tạo biến `orders`, tính giá trị đơn hàng trung bình `revenue / orders`, sau đó
in kết quả có dấu phân cách hàng nghìn và đơn vị đồng.

In [ ]:
orders = 420
average_order_value = revenue / orders
print(f"Giá trị đơn hàng trung bình: {average_order_value:,.0f} đ")
assert average_order_value > 0

## 2. List, dictionary, điều kiện và vòng lặp

`dictionary` phù hợp biểu diễn bộ KPI; vòng lặp giúp áp dụng cùng một quy tắc
cảnh báo cho nhiều chỉ số.

In [ ]:
kpis = {
    "revenue_growth": 0.08,
    "profit_margin": 0.172,
    "on_time_rate": 0.91,
}
targets = {
    "revenue_growth": 0.10,
    "profit_margin": 0.20,
    "on_time_rate": 0.95,
}

for name, value in kpis.items():
    label = "ĐẠT" if value >= targets[name] else "CẦN CHÚ Ý"
    print(f"{name:18s}: {value:6.1%} | {label}")

## 3. Hàm — đóng gói logic KPI để tái sử dụng

Hàm dưới đây được dùng lại ở Chương 2 khi tổng hợp theo vùng hoặc sản phẩm.

In [ ]:
def safe_margin(profit, revenue):
    # Trả về NaN nếu doanh thu bằng 0 để tránh phép chia không xác định.
    return np.nan if revenue == 0 else profit / revenue

assert np.isclose(safe_margin(20, 100), 0.2)
assert np.isnan(safe_margin(0, 0))
print(f"Biên lợi nhuận mẫu: {safe_margin(21_500_000, 125_000_000):.1%}")

## 4. DataFrame — nền tảng cho Chương 2

**Grain:** mỗi dòng là một giao dịch bán hàng. `region`, `month`, `category`
là dimension; `sales`, `profit` là measure.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
n = 120
sales = rng.integers(100_000, 2_000_000, n)
df = pd.DataFrame({
    "transaction_id": [f"GD{i:03d}" for i in range(1, n + 1)],
    "month": rng.choice(["2026-04", "2026-05", "2026-06"], n),
    "region": rng.choice(["Miền Bắc", "Miền Trung", "Miền Nam"], n),
    "category": rng.choice(["Đồ uống", "Thực phẩm", "Gia dụng"], n),
    "sales": sales,
    "profit": np.round(sales * rng.uniform(0.08, 0.32, n)).astype(int),
})
df.head()

In [ ]:
# Lọc → nhóm → tổng hợp → sắp xếp
south_summary = (df.loc[df["region"].eq("Miền Nam")]
                 .groupby("category", as_index=False)
                 .agg(sales=("sales", "sum"), profit=("profit", "sum"))
                 .sort_values("sales", ascending=False))
south_summary["margin"] = south_summary["profit"] / south_summary["sales"]
south_summary

### Bài tập 2

1. Lọc giao dịch tháng `2026-06`.
2. Tính doanh thu và lợi nhuận theo vùng.
3. Xác định vùng có doanh thu cao nhất.
4. Giải thích vì sao bảng này là **mô tả**, chưa phải **dự báo**.

In [ ]:
june = df[df["month"].eq("2026-06")]
june_by_region = (june.groupby("region", as_index=False)
                  .agg(sales=("sales", "sum"), profit=("profit", "sum"))
                  .sort_values("sales", ascending=False))
display(june_by_region)
print("Vùng doanh thu cao nhất:", june_by_region.iloc[0]["region"])

## 5. Trực quan cơ bản — cầu nối sang dashboard Chương 2

In [ ]:
monthly = df.groupby("month", as_index=False)["sales"].sum().sort_values("month")
ax = monthly.plot(x="month", y="sales", marker="o", legend=False, figsize=(7, 3.5))
ax.set(title="Doanh thu theo tháng", xlabel="Tháng", ylabel="Doanh thu (đ)")
plt.show()

## 6. Tạo biến và chia dữ liệu — chuẩn bị cho Chương 3

Ở đây **chưa huấn luyện mô hình**. Ta chỉ tạo biến mục tiêu và chia tập dữ
liệu để hiểu cấu trúc một bài toán dự báo.

In [ ]:
customers = pd.DataFrame({
    "tenure_months": rng.integers(1, 73, 100),
    "complaints": rng.poisson(0.7, 100),
    "monthly_fee": rng.integers(100_000, 600_000, 100),
})
customers["high_risk"] = (
    (customers["complaints"] >= 2) & (customers["tenure_months"] < 18)
).astype(int)

train = customers.sample(frac=0.8, random_state=RANDOM_SEED)
test = customers.drop(train.index)
print("Train:", train.shape, "| Test:", test.shape)
print("Tỷ lệ high_risk:", customers["high_risk"].mean().round(3))

**Câu hỏi kiểm tra:** Vì sao không được dùng tập test để điều chỉnh quy tắc/mô
hình? Biến `high_risk` ở ví dụ này chỉ là nhãn mô phỏng; trong thực tế cần định
nghĩa kết quả kinh doanh quan sát được.

## 7. Giá trị kỳ vọng và ràng buộc — chuẩn bị cho Chương 4

Một đề xuất không chỉ dựa vào xác suất. Ta cần kết hợp xác suất thành công,
lợi ích, chi phí và ngân sách.

In [ ]:
offers = pd.DataFrame({
    "option": ["Giảm giá", "Tặng dung lượng", "Chăm sóc cá nhân"],
    "success_probability": [0.55, 0.40, 0.68],
    "benefit_if_success": [500_000, 420_000, 650_000],
    "cost": [160_000, 80_000, 260_000],
})
offers["expected_net_value"] = (
    offers["success_probability"] * offers["benefit_if_success"] - offers["cost"]
)
offers.sort_values("expected_net_value", ascending=False)

### Bài tập tổng hợp

Với dữ liệu bán hàng ở phần 4:

1. Viết một câu hỏi mô tả cho Chương 2.
2. Nêu một biến mục tiêu có thể dự báo ở Chương 3.
3. Nêu hai phương án hành động và một ràng buộc cho Chương 4.
4. Chỉ rõ đầu ra Python nào sẽ hỗ trợ quyết định của nhà quản lý.

Mục tiêu là nhìn thấy vai trò liên tục của Python trong toàn bộ học phần,
không kết luận rằng một bảng thống kê tự nó đã là một quyết định.